### Prompts

Prompt allow you to create System Messages with input variables, for example, this:
**SystemMessage(content="You are a helpful assistant that translates the English to Spanish.")**
English and Spanish may be dynamic. This can be archieved with templates

In [1]:
! pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 2.7 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/sridba503/LangChain.git
%cd /content/LangChain
!pwd

Cloning into 'LangChain'...
remote: Enumerating objects: 590, done.
remote: Counting objects: 100% (590/590), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 590 (delta 284), reused 562 (delta 273), pack-reused 0 (from 0)
Receiving objects: 100% (590/590), 7.91 MiB | 22.02 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/LangChain
/content/LangChain


In [3]:
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

# Retrieve the API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Configure the Generative AI library with your API key
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [18]:
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(    model="gemini-3.1-flash-lite",   temperature=0.76, google_api_key=GOOGLE_API_KEY)
load_dotenv(find_dotenv())


True

In [19]:
TEMPLATE = """
You are a helpful assistant that translates the {input_language} to {output_language}
"""

In [20]:
from langchain_core.prompts.prompt import PromptTemplate

prompt_template = PromptTemplate.from_template(
    template=TEMPLATE
)
prompt_template.format(input_language="english", output_language="german")

'\nYou are a helpful assistant that translates the english to german\n'

In [23]:
# Define the input and output languages for translation
input_lang = "English"
output_lang = "French"

# Format the translation prompt using the prompt_template defined earlier
translation_prompt = prompt_template.format(input_language=input_lang, output_language=output_lang)

# Invoke the LLM with a message to translate
response = llm.invoke(translation_prompt + "\nTranslate: Hello, how are you?")
print(response.text)

Here is the translation in French:

**"Bonjour, comment allez-vous ?"** (formal) 
or 
**"Salut, comment ça va ?"** (informal)


Passing the input_variables to the constructor will provide additional validation for the template

In [ ]:
prompt_template = PromptTemplate(template=TEMPLATE, input_variables=["input_language", "output_language"])
prompt_template.format(input_language="english", output_language="german")


### Few Shot Prompt - provide a few examples in the template

In [ ]:
TEMPLATE = """
Interprete the text and evaluate the text.
sentiment: is the text in a positive, neutral or negative sentiment?
subject: What subject is the text about? Use exactly one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}
"""

To improve performance we can provide examples to increase the quality of the output.

In [25]:
TEMPLATE = """
Interprete the text and evaluate the text.
sentiment: is the text in a positive, neutral or negative sentiment?
subject: What subject is the text about? Use exactly one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}

Examples:
text: The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable.
sentiment: positive
subject: BellaVista

text: BellaVista restaurant was alright. The food was decent, but nothing stood out.
sentiment: neutral
subject: BellaVista

text: I was disappointed with BellaVista. The service was slow and the dishes lacked flavor.
sentiment: negative
subject: BellaVista

text: SeoulSavor offered the most authentic Korean flavors I've tasted outside of Seoul. The kimchi was perfectly fermented and spicy.
sentiment: positive
subject: SeoulSavor

text: SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste.
sentiment: neutral
subject: SeoulSavor

text: I didn't enjoy my meal at SeoulSavor. The tteokbokki was too mushy and the service was not attentive.
sentiment: negative
subject: SeoulSavor

text: MunichMeals has the best bratwurst and sauerkraut I've tasted outside of Bavaria. Their beer garden ambiance is truly authentic.
sentiment: positive
subject: MunichMeals

text: MunichMeals was alright. The weisswurst was okay, but I've had better elsewhere.
sentiment: neutral
subject: MunichMeals

text: I was let down by MunichMeals. The potato salad lacked flavor and the staff seemed uninterested.
sentiment: negative
subject: MunichMeals
"""


In [28]:

prompt_template = PromptTemplate(template=TEMPLATE, input_variables=["input"])
formatted_input = prompt_template.format(input="The MunichDeals experience was just awesome!")
response = llm.invoke(formatted_input)
print(response.text)

```json
{
  "sentiment": "positive",
  "subject": "MunichDeals"
}
```


LangChain also provides a FewShotPromptTemplate class, which allows creating the examples more modularized

In [29]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

examples = [
    {
        "text": "The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable.",
        "response": "sentiment: positive\nsubject: BellaVista"
    },
    {
        "text": "BellaVista restaurant was alright. The food was decent, but nothing stood out.",
        "response": "sentiment: neutral\nsubject: BellaVista"
    },
    ### other examples are left out
]

In [31]:
new_example = {
    "text": "SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste.",
    "response": "sentiment: neutral\nsubject: SeoulSavor"
}
examples.append(new_example)

In [32]:
# formatted_input = prompt_template.format(input="The MunichDeals experience was just awesome!")
response = llm.invoke(formatted_input)
print(response.text())

```json
{
  "sentiment": "positive",
  "subject": "MunichDeals"
}
```


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
example_prompt = PromptTemplate(input_variables=["text", "response"], template="Text: {text}\n{response}")

In [11]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="text: {input}",
    input_variables=["input"]
)

In [12]:
print(prompt.format(input="The MunichDeals experience was just awesome!"))

Text: The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable.
sentiment: positive
subject: BellaVista

Text: BellaVista restaurant was alright. The food was decent, but nothing stood out.
sentiment: neutral
subject: BellaVista

Text: SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste.
sentiment: neutral
subject: SeoulSavor

text: The MunichDeals experience was just awesome!


### Chain-of-thought Prompting

Instead of just providing examples, we can also provide examples which include the whole thought process of why a review is negative, neutral or positive

In [33]:
TEMPLATE = """
Interprete the text and evaluate the text. Determine if the text has a positive, neutral, or negative sentiment. Also, identify the subject of the text in one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}

Chain-of-Thought Prompts:
Let's start by evaluating a statement. Consider: "The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable." How does this make you feel about BellaVista?
 It sounds like a positive review for BellaVista.

Based on the positive nature of that statement, how would you format your response?
 { "sentiment": "positive", "subject": "BellaVista" }

Now, think about this: "SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste." Does this give a strong feeling either way?
 Not particularly. It seems like a mix of good and not-so-good elements, so it's neutral.

Given the neutral sentiment, how should this be presented?
 { "sentiment": "neutral", "subject": "SeoulSavor" }

Lastly, ponder on this: "I was let down by MunichMeals. The potato salad lacked flavor and the staff seemed uninterested." What's the overall impression here?
 The statement is expressing disappointment and dissatisfaction.

And if you were to categorize this impression, what would it be?
 { "sentiment": "negative", "subject": "MunichMeals" }
"""

### Serializing prompts

Example is an easy prompt since serializing does not work for the **PipelinePromptTemplate** (yet)

In [13]:
prompt = PromptTemplate(input_variables=["input"], template="Tell me a joke about {input}")
prompt.save("prompt.yaml")
prompt.save("prompt.json")

/tmp/ipykernel_2414/2098192752.py:2: LangChainDeprecationWarning: The method `BasePromptTemplate.save` was deprecated in langchain-core 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt.save("prompt.yaml")


In [14]:
from langchain_core.prompts import load_prompt

prompt = load_prompt("prompt.yaml")
prompt.format(input="Horses")

/tmp/ipykernel_2414/2014944641.py:3: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt("prompt.yaml")


'Tell me a joke about Horses'

In [16]:
from langchain_core.prompts import load_prompt

prompt = load_prompt("prompt.yaml")
print(prompt.format(input="horses"))

Tell me a joke about horses


In [15]:
prompt = load_prompt("prompt.json")
prompt.format(input="Donkeys")

'Tell me a joke about Donkeys'